In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import re

In [4]:
df = pd.read_csv('/content/Reviews.csv', quotechar='"', escapechar='\\',on_bad_lines='skip')

/tmp/ipython-input-141389736.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/Reviews.csv', quotechar='"', escapechar='\\',on_bad_lines='skip')


In [6]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1.0,5.0,1.303862e+09,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0.0,1.0,1.346976e+09,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1.0,4.0,1.219018e+09,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3.0,2.0,1.307923e+09,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0.0,5.0,1.350778e+09,Great taffy,Great taffy at a great price. There was a wid...


In [7]:
#Преобразовываем рейтинг из оценки в плохой/хороший
df['sentiment'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)  # 1=хороший, 0=плохой

# Очищаем текст
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Убираем спец символы
    text = re.sub(r'\s+', ' ', text).strip()  # Убираем лишние пробелы
    return text

df['cleaned_text'] = df['Text'].apply(clean_text)

In [8]:
# Разделяем данные
X = df['cleaned_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Токенезируем данные
vocab_size = 10000  # Указываем размер нашего словаря
max_length = 200    # Максимальное количество токенов в предложении
trunc_type = 'post' # Удаляем части предложений, которые идут после лимита
padding_type = 'post' # Добавляем текст в конец предложений что бы достич лимита
oov_tok = '<OOV>'   # Заменям слова, которых нет в словаре данным токеном

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

# Токенезируем текст
train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

# Добиваем размер предложений до нашего лимита
X_train_padded = pad_sequences(train_sequences, maxlen=max_length,
                              padding=padding_type, truncating=trunc_type)
X_test_padded = pad_sequences(test_sequences, maxlen=max_length,
                             padding=padding_type, truncating=trunc_type)

In [9]:
embedding_dim = 100

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Бинарная классификация
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Early stopping to prevent overfitting
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    X_train_padded, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test_padded, y_test),
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/20
14211/14211 ━━━━━━━━━━━━━━━━━━━━ 216s 15ms/step - accuracy: 0.8459 - loss: 0.3663 - precision: 0.8656 - recall: 0.9526 - val_accuracy: 0.8577 - val_loss: 0.3207 - val_precision: 0.8539 - val_recall: 0.9865
Epoch 2/20
14211/14211 ━━━━━━━━━━━━━━━━━━━━ 275s 16ms/step - accuracy: 0.8886 - loss: 0.2702 - precision: 0.9125 - recall: 0.9482 - val_accuracy: 0.8939 - val_loss: 0.2753 - val_precision: 0.8998 - val_recall: 0.9723
Epoch 3/20
14211/14211 ━━━━━━━━━━━━━━━━━━━━ 234s 16ms/step - accuracy: 0.8958 - loss: 0.2534 - precision: 0.9174 - recall: 0.9523 - val_accuracy: 0.9008 - val_loss: 0.2385 - val_precision: 0.9282 - val_recall: 0.9461
Epoch 4/20
14211/14211 ━━━━━━━━━━━━━━━━━━━━ 252s 16ms/step - accuracy: 0.8998 - loss: 0.2442 - precision: 0.9216 - recall: 0.9527 - val_accuracy: 0.8988 - val_loss: 0.2405 - val_precision: 0.9442 - val_recall: 0.9251
Epoch 5/20
14211/14211 ━━━━━━━━━━━━━━━━━━━━ 263s 16ms/step - accuracy: 0.9027 - loss: 0.2385 - precision: 0.9243 - recall: 0.9534 - 

KeyboardInterrupt: 

In [11]:
# Оцениваем тестовые данные
loss, accuracy, precision, recall = model.evaluate(X_test_padded, y_test)
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")

# Делаем предсказания
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")


print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))


print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.9089 - loss: 0.2333 - precision: 0.9288 - recall: 0.9567
Test Accuracy: 0.9094
Test Precision: 0.9284
Test Recall: 0.9578
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step

Classification Report:
              precision    recall  f1-score   support

    Negative       0.83      0.74      0.78     24933
    Positive       0.93      0.96      0.94     88752

    accuracy                           0.91    113685
   macro avg       0.88      0.85      0.86    113685
weighted avg       0.91      0.91      0.91    113685


Confusion Matrix:
[[18378  6555]
 [ 3749 85003]]


In [12]:
# Строим график обучения
def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()

    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()

    plt.tight_layout()
    plt.show()

plot_training_history(history)

NameError: name 'history' is not defined